## 2026 NFL Fantasy WR Rankings Project Part 1 - Data Scraping and Cleaning

### Imports

In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
import nflreadpy as nfl

In [3]:
years = list(range(2020,2026))

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
pd.set_option('display.max_rows', None)

### Scraping from NFL API

In [6]:
weekly_stats = nfl.load_player_stats(years)

In [96]:
weekly = weekly_stats.to_pandas()

In [8]:
print(list(weekly.columns))

['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving_16', 'receivi

**Cleaning:**
- Filter to just regular season with 'season_type'
- Standardize names (remove Jr., Sr., II, III)
- Need to aggregate weekly stats into seasons

### Data Cleaning and Manipulation

In [97]:
weekly = weekly[weekly['season_type'] == 'REG']

In [98]:
weekly['player_display_name'] = (
    weekly['player_display_name']
    .str.replace(r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$', '', regex=True)
)

In [99]:
metrics_to_average = [ 
    'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 
    'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 
    'passing_first_downs', 'passing_2pt_conversions', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 
    'passing_epa', 'passing_cpoe', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 
    'rushing_first_downs', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 
    'rushing_20', 'rushing_40', 'rushing_epa','receptions', 'targets', 'receiving_yards', 'receiving_tds', 
    'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 
    'receiving_first_downs', 'receiving_2pt_conversions', 'receiving_10', 'receiving_16', 
    'receiving_20', 'receiving_40', 'receiving_epa' 'target_share', 'air_yards_share', 'wopr', 
    'special_teams_tds', 'punt_returns', 'punt_return_yards', 'kickoff_returns', 
    'kickoff_return_yards', 'fantasy_points', 'fantasy_points_ppr'
]

In the above, some stats like passing_cpoe (completion percentage over expectation) would ideally be recalculated on season sums, but I don't have the necessary data to do that so I am performing a simple average as a proxy.

I am not including the following columns below because they are essentially calculations made using other fields already in my dataset, which will have a high correlation.
- PACR: passing yards / passing air yards
- RACR: receiving yards / receiving air yards


In [100]:
# 1. Clean up the columns: Identify which metrics are actually in your weekly DataFrame
valid_metrics = [col for col in metrics_to_average if col in weekly.columns]

# 2. Automatically find and convert any numeric columns that are currently stored as 'str'
for col in valid_metrics:
    # If the column is numeric-like but pandas thinks it's a string, convert it to float
    if weekly[col].dtype in ['string', 'object', 'str']:
        # errors='coerce' turns non-numeric values (like empty strings or lists) into NaN
        weekly[col] = pd.to_numeric(weekly[col], errors='coerce')

In [101]:
numeric_metrics = []
for col in valid_metrics:
    if pd.api.types.is_numeric_dtype(weekly[col]):
        numeric_metrics.append(col)
    else:
        print(f"Skipping non-numeric column: {col} (dtype: {weekly[col].dtype})")

In [102]:
# 4. Build the aggregation dictionary for the numeric columns only
agg_dict = {f"{metric}_pg": (metric, 'mean') for metric in numeric_metrics}

In [103]:
seasonal = weekly.groupby(['player_display_name', 'season']).agg(
    **agg_dict,
    games_played=('week', 'count')
).reset_index()

In [15]:
seasonal[seasonal['player_display_name'] == 'Brian Robinson']

,player_display_name,season,completions_pg,attempts_pg,passing_yards_pg,passing_tds_pg,passing_interceptions_pg,sacks_suffered_pg,sack_yards_lost_pg,sack_fumbles_pg,sack_fumbles_lost_pg,passing_air_yards_pg,passing_yards_after_catch_pg,passing_first_downs_pg,passing_2pt_conversions_pg,passing_10_pg,passing_16_pg,passing_20_pg,passing_40_pg,passing_epa_pg,passing_cpoe_pg,carries_pg,rushing_yards_pg,rushing_tds_pg,rushing_fumbles_pg,rushing_fumbles_lost_pg,rushing_first_downs_pg,rushing_2pt_conversions_pg,rushing_10_pg,rushing_12_pg,rushing_20_pg,rushing_40_pg,rushing_epa_pg,receptions_pg,targets_pg,receiving_yards_pg,receiving_tds_pg,receiving_fumbles_pg,receiving_fumbles_lost_pg,receiving_air_yards_pg,receiving_yards_after_catch_pg,receiving_first_downs_pg,receiving_2pt_conversions_pg,receiving_10_pg,receiving_16_pg,receiving_20_pg,receiving_40_pg,air_yards_share_pg,wopr_pg,special_teams_tds_pg,punt_returns_pg,punt_return_yards_pg,kickoff_returns_pg,kickoff_return_yards_pg,fantasy_points_pg,fantasy_points_ppr_pg,games_played
1376,Brian Robinson,2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,17.083333,66.416667,0.166667,0.166667,0.000000,4.083333,0.000000,1.500000,0.916667,0.166667,0.000000,-0.519616,0.750000,1.000000,5.000000,0.083333,0.000000,0.0,-0.916667,6.000000,0.250000,0.0,0.250000,0.083333,0.000000,0.000000,-0.004087,0.050633,0.0,0.0,0.0,0.000000,0.000000,8.641667,9.391667,12
1377,Brian Robinson,2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,11.866667,48.866667,0.333333,0.200000,0.133333,3.133333,0.066667,1.266667,0.866667,0.266667,0.000000,-1.409103,2.400000,2.866667,24.533333,0.266667,0.066667,0.0,-3.666667,27.333333,1.000000,0.0,0.866667,0.733333,0.533333,0.133333,-0.012869,0.109573,0.0,0.0,0.0,0.000000,0.000000,10.806667,13.206667,15
1378,Brian Robinson,2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,13.357143,57.071429,0.571429,0.142857,0.142857,3.357143,0.000000,1.357143,0.857143,0.285714,0.142857,-0.862933,1.428571,1.785714,11.357143,0.000000,0.000000,0.0,-1.642857,12.785714,0.285714,0.0,0.285714,0.071429,0.071429,0.000000,-0.009559,0.082987,0.0,0.0,0.0,0.000000,0.000000,9.985714,11.414286,14
1379,Brian Robinson,2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,5.411765,23.529412,0.117647,0.000000,0.000000,1.176471,0.000000,0.588235,0.352941,0.000000,0.000000,-0.047974,0.470588,0.705882,1.470588,0.000000,0.000000,0.0,0.000000,2.058824,0.058824,0.0,0.000000,0.000000,0.000000,0.000000,-0.000047,0.031288,0.0,0.0,0.0,0.764706,22.235294,3.205882,3.676471,17


In [59]:
seasonal.columns

Index(['player_display_name', 'season', 'completions_pg', 'attempts_pg',
       'passing_yards_pg', 'passing_tds_pg', 'passing_interceptions_pg',
       'sacks_suffered_pg', 'sack_yards_lost_pg', 'sack_fumbles_pg',
       'sack_fumbles_lost_pg', 'passing_air_yards_pg',
       'passing_yards_after_catch_pg', 'passing_first_downs_pg',
       'passing_2pt_conversions_pg', 'passing_10_pg', 'passing_16_pg',
       'passing_20_pg', 'passing_40_pg', 'passing_epa_pg', 'passing_cpoe_pg',
       'carries_pg', 'rushing_yards_pg', 'rushing_tds_pg',
       'rushing_fumbles_pg', 'rushing_fumbles_lost_pg',
       'rushing_first_downs_pg', 'rushing_2pt_conversions_pg', 'rushing_10_pg',
       'rushing_12_pg', 'rushing_20_pg', 'rushing_40_pg', 'rushing_epa_pg',
       'receptions_pg', 'targets_pg', 'receiving_yards_pg', 'receiving_tds_pg',
       'receiving_fumbles_pg', 'receiving_fumbles_lost_pg',
       'receiving_air_yards_pg', 'receiving_yards_after_catch_pg',
       'receiving_first_downs_pg', 're

**Teammate Dataset**

The cleaning is complete. I am now creating duplicates of these statistical dataframes because a part of my model will include the stats from the player's best teammate.

In [104]:
teammate_seasonal = seasonal.copy()

In [105]:
teammate_seasonal.columns = 'Teammate_' + teammate_seasonal.columns

In [106]:
passing_cols = ['Teammate_completions_pg', 'Teammate_attempts_pg',
       'Teammate_passing_yards_pg', 'Teammate_passing_tds_pg',
       'Teammate_passing_interceptions_pg', 'Teammate_sacks_suffered_pg',
       'Teammate_sack_yards_lost_pg', 'Teammate_sack_fumbles_pg',
       'Teammate_sack_fumbles_lost_pg', 'Teammate_passing_air_yards_pg',
       'Teammate_passing_yards_after_catch_pg',
       'Teammate_passing_first_downs_pg',
       'Teammate_passing_2pt_conversions_pg', 'Teammate_passing_10_pg',
       'Teammate_passing_16_pg', 'Teammate_passing_20_pg',
       'Teammate_passing_40_pg', 'Teammate_passing_epa_pg',
       'Teammate_passing_cpoe_pg', 'Teammate_fantasy_points_pg',
]

In [107]:
teammate_seasonal = teammate_seasonal.drop(passing_cols, axis=1)

In [108]:
teammate_seasonal[teammate_seasonal['Teammate_player_display_name'] == 'Jalen Coker']

,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
5185,Jalen Coker,2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2.909091,4.181818,43.454545,0.181818,0.000000,0.000000,46.909091,15.545455,2.090909,0.000000,1.636364,0.818182,0.545455,0.090909,0.201400,0.362436,0.0,0.0,0.0,0.0,0.0,8.414545,11
5186,Jalen Coker,2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.000000,3.909091,35.818182,0.272727,0.090909,0.090909,39.545455,9.272727,2.000000,0.090909,1.454545,0.818182,0.545455,0.000000,0.199885,0.370905,0.0,0.0,0.0,0.0,0.0,8.218182,11


In [58]:
teammate_seasonal.columns

Index(['Teammate_player_display_name', 'Teammate_season',
       'Teammate_carries_pg', 'Teammate_rushing_yards_pg',
       'Teammate_rushing_tds_pg', 'Teammate_rushing_fumbles_pg',
       'Teammate_rushing_fumbles_lost_pg', 'Teammate_rushing_first_downs_pg',
       'Teammate_rushing_2pt_conversions_pg', 'Teammate_rushing_10_pg',
       'Teammate_rushing_12_pg', 'Teammate_rushing_20_pg',
       'Teammate_rushing_40_pg', 'Teammate_rushing_epa_pg',
       'Teammate_receptions_pg', 'Teammate_targets_pg',
       'Teammate_receiving_yards_pg', 'Teammate_receiving_tds_pg',
       'Teammate_receiving_fumbles_pg', 'Teammate_receiving_fumbles_lost_pg',
       'Teammate_receiving_air_yards_pg',
       'Teammate_receiving_yards_after_catch_pg',
       'Teammate_receiving_first_downs_pg',
       'Teammate_receiving_2pt_conversions_pg', 'Teammate_receiving_10_pg',
       'Teammate_receiving_16_pg', 'Teammate_receiving_20_pg',
       'Teammate_receiving_40_pg', 'Teammate_air_yards_share_pg',
       

In [41]:
teammate_seasonal.to_csv("../outputs/teammate_season_stats_2020_2025.csv", index=False)

In [42]:
teammate_seasonal = pd.read_csv('../outputs/teammate_season_stats_2020_2025.csv')

**Quarterback Dataset**

I am now creating one more duplicate of these statistical dataframes because a part of my model will include the stats from the player's quarterback. Because of that I will be pulling from the rushing and receiving dataframes twice: once for the rookie's stats and once for the teammate's stats.

In [109]:
qb_seasonal = seasonal.copy()

In [110]:
qb_seasonal.columns = 'QB_' + qb_seasonal.columns

In [111]:
non_passing_cols = ['QB_carries_pg', 'QB_rushing_yards_pg',
       'QB_rushing_tds_pg', 'QB_rushing_fumbles_pg',
       'QB_rushing_fumbles_lost_pg', 'QB_rushing_first_downs_pg',
       'QB_rushing_2pt_conversions_pg', 'QB_rushing_10_pg', 'QB_rushing_12_pg',
       'QB_rushing_20_pg', 'QB_rushing_40_pg', 'QB_rushing_epa_pg',
       'QB_receptions_pg', 'QB_targets_pg', 'QB_receiving_yards_pg',
       'QB_receiving_tds_pg', 'QB_receiving_fumbles_pg',
       'QB_receiving_fumbles_lost_pg', 'QB_receiving_air_yards_pg',
       'QB_receiving_yards_after_catch_pg', 'QB_receiving_first_downs_pg',
       'QB_receiving_2pt_conversions_pg', 'QB_receiving_10_pg',
       'QB_receiving_16_pg', 'QB_receiving_20_pg', 'QB_receiving_40_pg',
       'QB_air_yards_share_pg', 'QB_wopr_pg', 'QB_special_teams_tds_pg',
       'QB_punt_returns_pg', 'QB_punt_return_yards_pg',
       'QB_kickoff_returns_pg', 'QB_kickoff_return_yards_pg',
       'QB_fantasy_points_pg',
]

In [112]:
qb_seasonal = qb_seasonal.drop(non_passing_cols, axis=1)

In [25]:
qb_seasonal[qb_seasonal['QB_player_display_name'] == 'Tyrod Taylor']

,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played
11513,Tyrod Taylor,2020,8.000000,15.000000,104.000000,0.000000,0.000000,1.000000,-0.500000,0.000000,0.000000,167.500000,40.000000,5.000000,0.0,4.500000,2.000000,1.500000,0.000000,-0.191233,-2.468618,4.510000,2
11514,Tyrod Taylor,2021,15.166667,25.000000,161.000000,0.833333,0.833333,2.166667,-19.833333,0.000000,0.000000,184.500000,81.833333,7.333333,0.0,6.166667,2.333333,2.166667,0.500000,-2.479269,-4.032346,13.623333,6
11515,Tyrod Taylor,2022,3.000000,4.000000,29.000000,0.500000,0.500000,1.500000,-9.000000,0.500000,0.500000,49.500000,13.000000,1.500000,0.5,1.500000,0.500000,0.000000,0.000000,-1.666168,2.872760,5.660000,2
11516,Tyrod Taylor,2023,11.600000,18.000000,134.100000,0.500000,0.300000,1.700000,-9.400000,0.200000,0.000000,154.100000,57.800000,5.200000,0.0,4.500000,2.900000,1.900000,0.400000,0.116902,-0.614916,8.734000,10
11517,Tyrod Taylor,2024,8.500000,11.000000,59.500000,1.500000,0.000000,0.000000,0.000000,0.000000,0.000000,44.000000,34.000000,4.500000,0.5,1.500000,0.500000,0.500000,0.000000,5.531543,12.303555,10.030000,2
11518,Tyrod Taylor,2025,13.333333,22.333333,129.833333,0.833333,0.833333,2.333333,-13.666667,0.166667,0.166667,190.833333,55.166667,6.833333,0.0,5.000000,1.500000,1.000000,0.333333,-2.219152,-7.356643,9.910000,6


In [57]:
qb_seasonal.columns

Index(['QB_player_display_name', 'QB_season', 'QB_completions_pg',
       'QB_attempts_pg', 'QB_passing_yards_pg', 'QB_passing_tds_pg',
       'QB_passing_interceptions_pg', 'QB_sacks_suffered_pg',
       'QB_sack_yards_lost_pg', 'QB_sack_fumbles_pg',
       'QB_sack_fumbles_lost_pg', 'QB_passing_air_yards_pg',
       'QB_passing_yards_after_catch_pg', 'QB_passing_first_downs_pg',
       'QB_passing_2pt_conversions_pg', 'QB_passing_10_pg', 'QB_passing_16_pg',
       'QB_passing_20_pg', 'QB_passing_40_pg', 'QB_passing_epa_pg',
       'QB_passing_cpoe_pg', 'QB_fantasy_points_ppr_pg', 'QB_games_played'],
      dtype='str')

In [32]:
qb_seasonal.to_csv("../outputs/qb_season_stats_2020_2025.csv", index=False)

In [33]:
qb_seasonal = pd.read_csv('../outputs/qb_season_stats_2020_2025.csv')

### Player Dataset

the previous season data for the player himself

In [113]:
player_previous_seasonal = seasonal.copy()

In [114]:
player_previous_seasonal.columns = 'Previous_' + player_previous_seasonal.columns

In [115]:
passing_cols = ['Previous_completions_pg', 'Previous_attempts_pg',
       'Previous_passing_yards_pg', 'Previous_passing_tds_pg',
       'Previous_passing_interceptions_pg', 'Previous_sacks_suffered_pg',
       'Previous_sack_yards_lost_pg', 'Previous_sack_fumbles_pg',
       'Previous_sack_fumbles_lost_pg', 'Previous_passing_air_yards_pg',
       'Previous_passing_yards_after_catch_pg',
       'Previous_passing_first_downs_pg',
       'Previous_passing_2pt_conversions_pg', 'Previous_passing_10_pg',
       'Previous_passing_16_pg', 'Previous_passing_20_pg',
       'Previous_passing_40_pg', 'Previous_passing_epa_pg',
       'Previous_passing_cpoe_pg',  'Previous_fantasy_points_pg',
]

In [116]:
player_previous_seasonal = player_previous_seasonal.drop(passing_cols, axis=1)

In [30]:
player_previous_seasonal[player_previous_seasonal['Previous_player_display_name'] == 'Odell Beckham']

,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played
9085,Odell Beckham,2020,0.428571,10.285714,0.142857,0.0,0.0,0.285714,0.0,0.285714,0.285714,0.285714,0.142857,2.527460,3.285714,6.142857,45.571429,0.428571,0.000000,0.000000,83.428571,6.571429,2.428571,0.0,2.142857,1.000000,0.428571,0.142857,0.333189,0.563187,0.0,0.0,0.0,0.0,0.0,12.402857,7
9086,Odell Beckham,2021,0.142857,1.000000,0.000000,0.0,0.0,0.071429,0.0,0.071429,0.000000,0.000000,0.000000,0.232988,3.142857,5.857143,38.357143,0.357143,0.000000,0.000000,78.000000,10.785714,1.928571,0.0,1.357143,1.000000,0.500000,0.142857,0.260737,0.449264,0.0,0.0,0.0,0.0,0.0,9.221429,14
9087,Odell Beckham,2023,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,NaN,2.500000,4.571429,40.357143,0.214286,0.071429,0.071429,63.428571,11.071429,1.928571,0.0,1.571429,0.785714,0.714286,0.214286,0.243101,0.406095,0.0,0.0,0.0,0.0,0.0,7.678571,14
9088,Odell Beckham,2024,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,NaN,1.125000,2.250000,6.875000,0.000000,0.000000,0.000000,17.000000,1.750000,0.375000,0.0,0.125000,0.000000,0.000000,0.000000,0.080263,0.161314,0.0,0.0,0.0,0.0,0.0,1.812500,8


In [56]:
player_previous_seasonal.columns

Index(['Previous_player_display_name', 'Previous_season',
       'Previous_carries_pg', 'Previous_rushing_yards_pg',
       'Previous_rushing_tds_pg', 'Previous_rushing_fumbles_pg',
       'Previous_rushing_fumbles_lost_pg', 'Previous_rushing_first_downs_pg',
       'Previous_rushing_2pt_conversions_pg', 'Previous_rushing_10_pg',
       'Previous_rushing_12_pg', 'Previous_rushing_20_pg',
       'Previous_rushing_40_pg', 'Previous_rushing_epa_pg',
       'Previous_receptions_pg', 'Previous_targets_pg',
       'Previous_receiving_yards_pg', 'Previous_receiving_tds_pg',
       'Previous_receiving_fumbles_pg', 'Previous_receiving_fumbles_lost_pg',
       'Previous_receiving_air_yards_pg',
       'Previous_receiving_yards_after_catch_pg',
       'Previous_receiving_first_downs_pg',
       'Previous_receiving_2pt_conversions_pg', 'Previous_receiving_10_pg',
       'Previous_receiving_16_pg', 'Previous_receiving_20_pg',
       'Previous_receiving_40_pg', 'Previous_air_yards_share_pg',
       

In [54]:
player_previous_seasonal.to_csv("../outputs/player_season_stats_2020_2025.csv", index=False)

In [55]:
player_previous_seasonal = pd.read_csv('../outputs/player_season_stats_2020_2025.csv')

### Target Dataset

In [60]:
seasonal.columns

Index(['player_display_name', 'season', 'completions_pg', 'attempts_pg',
       'passing_yards_pg', 'passing_tds_pg', 'passing_interceptions_pg',
       'sacks_suffered_pg', 'sack_yards_lost_pg', 'sack_fumbles_pg',
       'sack_fumbles_lost_pg', 'passing_air_yards_pg',
       'passing_yards_after_catch_pg', 'passing_first_downs_pg',
       'passing_2pt_conversions_pg', 'passing_10_pg', 'passing_16_pg',
       'passing_20_pg', 'passing_40_pg', 'passing_epa_pg', 'passing_cpoe_pg',
       'carries_pg', 'rushing_yards_pg', 'rushing_tds_pg',
       'rushing_fumbles_pg', 'rushing_fumbles_lost_pg',
       'rushing_first_downs_pg', 'rushing_2pt_conversions_pg', 'rushing_10_pg',
       'rushing_12_pg', 'rushing_20_pg', 'rushing_40_pg', 'rushing_epa_pg',
       'receptions_pg', 'targets_pg', 'receiving_yards_pg', 'receiving_tds_pg',
       'receiving_fumbles_pg', 'receiving_fumbles_lost_pg',
       'receiving_air_yards_pg', 'receiving_yards_after_catch_pg',
       'receiving_first_downs_pg', 're

In [117]:
target = seasonal.copy()

In [118]:
non_target_cols = ['completions_pg', 'attempts_pg',
       'passing_yards_pg', 'passing_tds_pg', 'passing_interceptions_pg',
       'sacks_suffered_pg', 'sack_yards_lost_pg', 'sack_fumbles_pg',
       'sack_fumbles_lost_pg', 'passing_air_yards_pg',
       'passing_yards_after_catch_pg', 'passing_first_downs_pg',
       'passing_2pt_conversions_pg', 'passing_10_pg', 'passing_16_pg',
       'passing_20_pg', 'passing_40_pg', 'passing_epa_pg', 'passing_cpoe_pg',
       'carries_pg', 'rushing_yards_pg', 'rushing_tds_pg',
       'rushing_fumbles_pg', 'rushing_fumbles_lost_pg',
       'rushing_first_downs_pg', 'rushing_2pt_conversions_pg', 'rushing_10_pg',
       'rushing_12_pg', 'rushing_20_pg', 'rushing_40_pg', 'rushing_epa_pg',
       'receptions_pg', 'targets_pg', 'receiving_yards_pg', 'receiving_tds_pg',
       'receiving_fumbles_pg', 'receiving_fumbles_lost_pg',
       'receiving_air_yards_pg', 'receiving_yards_after_catch_pg',
       'receiving_first_downs_pg', 'receiving_2pt_conversions_pg',
       'receiving_10_pg', 'receiving_16_pg', 'receiving_20_pg',
       'receiving_40_pg', 'air_yards_share_pg', 'wopr_pg',
       'special_teams_tds_pg', 'punt_returns_pg', 'punt_return_yards_pg',
       'kickoff_returns_pg', 'kickoff_return_yards_pg', 'fantasy_points_pg', 'games_played'
]

In [119]:
target = target.drop(non_target_cols, axis=1)

In [34]:
target[target['player_display_name'] == 'Will Fuller']

,player_display_name,season,fantasy_points_ppr_pg
11684,Will Fuller,2020,17.172727
11685,Will Fuller,2021,4.300000


In [65]:
target.columns

Index(['player_display_name', 'season', 'fantasy_points_ppr_pg'], dtype='str')

In [66]:
target.to_csv("../outputs/player_fppg_stats_2020_2025.csv", index=False)

In [67]:
target = pd.read_csv('../outputs/player_fppg_stats_2020_2025.csv')

### Manual Input Dataset - WR

This dataset was put together from Claude and QA'd. It has the top 60 non-rookie WRs from 2021-2026, as well as that player's preseason projected QB and projected best WR teammate as well as their ages.

In [117]:
fantasy_board = pd.read_csv('../inputs/top60_preseason_wr_ppr_2021_2026.csv')

In [118]:
fantasy_board.head()

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age
0,1,Davante Adams,2021,2020,28,GB,Aaron Rodgers,2020,37,Allen Lazard,2020,26
1,2,Tyreek Hill,2021,2020,27,KC,Patrick Mahomes,2020,25,Mecole Hardman,2020,23
2,3,Stefon Diggs,2021,2020,27,BUF,Josh Allen,2020,25,Gabe Davis,2020,22
3,4,DeAndre Hopkins,2021,2020,29,ARI,Kyler Murray,2020,24,A.J. Green,2020,33
4,5,Calvin Ridley,2021,2020,26,ATL,Matt Ryan,2020,36,Russell Gage,2020,25


In [119]:
fantasy_board['player_name'] = (
    fantasy_board['player_name']
    .str.replace(r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$', '', regex=True)
)

fantasy_board['team_qb'] = (
    fantasy_board['team_qb']
    .str.replace(r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$', '', regex=True)
)

fantasy_board['teammate'] = (
    fantasy_board['teammate']
    .str.replace(r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$', '', regex=True)
)

In [120]:
temp_wr_1 = pd.merge(fantasy_board, target, left_on=['player_name', 'season'], 
                   right_on=['player_display_name', 'season'], how='left')

In [121]:
temp_wr_2 = pd.merge(temp_wr_1, player_previous_seasonal, left_on=['player_name', 'past_season'], 
                   right_on=['Previous_player_display_name', 'Previous_season'], how='left')

In [122]:
temp_wr_3 = pd.merge(temp_wr_2, qb_seasonal, left_on=['team_qb', 'team_qb_season'], 
                   right_on=['QB_player_display_name', 'QB_season'], how='left')

In [123]:
final_wr = pd.merge(temp_wr_3, teammate_seasonal, left_on=['teammate', 'teammate_season'], 
                   right_on=['Teammate_player_display_name', 'Teammate_season'], how='left')

In [124]:
final_wr.head()

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
0,1,Davante Adams,2021,2020,28,GB,Aaron Rodgers,2020,37,Allen Lazard,2020,26,Davante Adams,21.518750,Davante Adams,2020.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,NaN,8.214286,10.642857,98.142857,1.285714,0.071429,0.071429,95.000000,42.642857,5.214286,0.000000,3.500000,1.785714,1.285714,0.357143,0.383416,0.778455,0.0,0.000000,0.0,0.0,0.0,25.600000,14.0,Aaron Rodgers,2020,23.2500,32.875,268.6875,3.000000,0.3125,1.250000,-11.3750,0.0625,0.062500,258.4375,140.5000,13.500000,0.0000,9.750000,4.8125,3.562500,0.875000,11.640741,7.353442,23.95375,16,Allen Lazard,2020,0.2000,1.7000,0.0,0.0,0.0,0.1000,0.0,0.1000,0.100,0.0000,0.0,0.197563,3.3000,4.6000,45.1000,0.3000,0.0000,0.0000,45.8000,19.5000,2.3000,0.0000,1.5000,0.6000,0.6000,0.2000,0.187132,0.364185,0.0000,0.0000,0.0,0.0000,0.0,9.78000,10
1,2,Tyreek Hill,2021,2020,27,KC,Patrick Mahomes,2020,25,Mecole Hardman,2020,23,Tyreek Hill,17.441176,Tyreek Hill,2020.0,0.866667,8.200000,0.133333,0.0,0.0,0.333333,0.0,0.266667,0.2,0.2,0.0,0.820106,5.800000,9.000000,85.066667,1.000000,0.000000,0.000000,115.933333,28.933333,3.800000,0.000000,3.266667,1.800000,1.333333,0.333333,0.347848,0.592705,0.0,0.066667,0.0,0.0,0.0,21.926667,15.0,Patrick Mahomes,2020,26.0000,39.200,316.0000,2.533333,0.4000,1.466667,-9.8000,0.2000,0.133333,327.8000,149.4000,15.86

**Data Cleaning**

Find players that did not have joined data because of formatting issues in names (capitalization, apostrophes, etc.) and change the name in the manual csv file. 

Once done, players with player_display_name NaN mean that they did not record any stats that season. Impute a zero for fantasy_points_ppr_pg.

Same with teammate_player_display_name.

Then rerun the isnull code and impute zeroes for anything remaining (not including fantasy points per game for 2026 rookies).

In [125]:
# View all rows where player_display_name is null
null_players_target = final_wr[final_wr['player_display_name'].isna()]

null_players_target.head(10)

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
263,48,Brandon Aiyuk,2025,2024,27,SF,Brock Purdy,2024,25,Ricky Pearsall,2024,24,NaN,NaN,Brandon Aiyuk,2024.0,0.000000,0.000000,0.0000,0.0,0.0,0.000000,0.0,0.000000,0.0000,0.0000,0.0000,NaN,3.571429,6.714286,53.428571,0.000000,0.000000,0.000000,79.428571,15.285714,2.714286,0.000000,2.428571,1.142857,0.714286,0.142857,0.267098,0.519892,0.0,0.000000,0.0,0.0,0.0,8.914286,7.0,Brock Purdy,2024,20.000000,30.333333,257.600000,1.333333,0.800000,2.066667,-10.400000,0.333333,0.200000,260.266667,109.600000,11.866667,0.000000,10.400000,5.066667,3.733333,0.733333,5.185512,1.877466,17.790667,15,Ricky Pearsall,2024,0.272727,4.090909,0.0,0.0,0.0,0.090909,0.0,0.090909,0.090909,0.090909,0.0,1.103091,2.818182,4.181818,36.363636,0.272727,0.000000,0.000000,48.000000,10.363636,1.727273,0.000000,1.363636,0.545455,0.454545,0.181818,0.183686,0.320898,0.000000,0.272727,3.727273,0.000000,0.000000,8.500000,11
268,53,Diontae Johnson,2025,2024,29,BAL,Lamar Jackson,2024,28,Zay Flowers,2024,25,NaN,NaN,Diontae Johnson,2024.0,0.181818,0.545455,0.0000,0.0,0.0,0.090909,0.0,0.000000,0.0000,0.0000,0.0000,2.038883,3.000000,6.090909,34.090909,0.272727,0.000000,0.000000,66.454545,8.272727,1.909091,0.000000,1.545455,0.909091,0.454545,0.000000,0.324116,0.511485,0.0,0.090909,0.0,0.0,0.0,8.100000,11.0,Lamar Jackson,2024,18.588235,27.882353,245.411765

In [98]:
seasonal[seasonal['player_display_name'] == "DJ Moore"]

,player_display_name,season,completions_pg,attempts_pg,passing_yards_pg,passing_tds_pg,passing_interceptions_pg,sacks_suffered_pg,sack_yards_lost_pg,sack_fumbles_pg,sack_fumbles_lost_pg,passing_air_yards_pg,passing_yards_after_catch_pg,passing_first_downs_pg,passing_2pt_conversions_pg,passing_10_pg,passing_16_pg,passing_20_pg,passing_40_pg,passing_epa_pg,passing_cpoe_pg,carries_pg,rushing_yards_pg,rushing_tds_pg,rushing_fumbles_pg,rushing_fumbles_lost_pg,rushing_first_downs_pg,rushing_2pt_conversions_pg,rushing_10_pg,rushing_12_pg,rushing_20_pg,rushing_40_pg,rushing_epa_pg,receptions_pg,targets_pg,receiving_yards_pg,receiving_tds_pg,receiving_fumbles_pg,receiving_fumbles_lost_pg,receiving_air_yards_pg,receiving_yards_after_catch_pg,receiving_first_downs_pg,receiving_2pt_conversions_pg,receiving_10_pg,receiving_16_pg,receiving_20_pg,receiving_40_pg,air_yards_share_pg,wopr_pg,special_teams_tds_pg,punt_returns_pg,punt_return_yards_pg,kickoff_returns_pg,kickoff_return_yards_pg,fantasy_points_pg,fantasy_points_ppr_pg,games_played
2660,DJ Moore,2020,0.000000,0.066667,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.400000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,-0.408471,-74.807233,0.133333,1.466667,0.000000,0.0,0.0,0.066667,0.0,0.066667,0.066667,0.066667,0.0,0.484478,4.400000,7.866667,79.533333,0.266667,0.000000,0.000000,104.066667,25.466667,3.533333,0.000000,3.266667,1.933333,1.266667,0.400000,0.404024,0.651482,0.0,0.000000,0.000000,0.000000,0.000000,9.700000,14.100000,15
2661,DJ Moore,2021,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.470588,2.823529,0.000000,0.0,0.0,0.235294,0.0,0.176471,0.117647,0.000000,0.0,1.058244,5.470588,9.588235,68.058824,0.235294,0.058824,0.058824,102.000000,24.823529,3.529412,0.058824,3.058824,1.647059,0.941176,0.058824,0.408123,0.712635,0.0,0.058824,0.529412,0.058824,0.764706,8.500000,13.970588,17
2662,DJ Moore,2022,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.588235,3.117647,0.000000,0.0,0.0,0.176471,0.0,0.058824,0.000000,0.000000,0.0,0.229414,3.705882,6.941176,52.235294,0.411765,0.000000,0.000000,89.764706,10.941176,2.588235,0.000000,2.117647,1.352941,1.000000,0.235294,0.531769,0.804432,0.0,0.000000,0.000000,0.000000,0.000000,8.005882,11.711765,17
2663,DJ Moore,2023,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.235294,1.235294,0.058824,0.0,0.0,0.058824,0.0,0.058824,0.058824,0.000000,0.0,-0.806124,5.647059,8.000000,80.235294,0.470588,0.058824,0.058824,86.823529,31.705882,3.764706,0.000000,3.117647,1.882353,1.470588,0.117647,0.401385,0.723262,0.0,0.000000,0.000000,0.000000,0.000000,11.205882,16.852941,17
2664,DJ Moore,2024,0.000000,0.058824,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,1.941176,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,-0.447272,-29.839304,0.823529,4.411765,0.000000,0.0,0.0,0.176471,0.0,0.176471,0.117647,0.000000,0.0,0.137396,5.764706,8.235294,56.823529,0.352941,0.058824,0.058824,61.705882,34.588235,2.529412,0.058824,2.352941,1.176471,0.647059,0.058824,0.238476,0.573303,0.0,0.000000,0.000000,0.000000,0.000000,8.241176,14.005882,17
2665,DJ Moore,2025,0.058824,0.058824,0.117647,0.058824,0.0,0.0,0.0,0.0,0.0,0.117647,0.0,0.058824,0.0,0.0,0.0,0.0,0.0,3.868194,46.214199,0.882353,4.647059,0.058824,0.0,0.0,0.294118,0.0,0.176471,0.117647,0.000000,0.0,0.463648,2.941176,5.000000,40.117647,0.352941,0.000000,0.000000,58.117647,13.058824,1.882353,0.000000,1.823529,1.117647,0.705882,0.117647,0.198831,0.381192,0.0,0.000000,0.000000,0.000000,0.000000,7.187059,10.128235,17


In [126]:
# View all rows where player_display_name is null
null_players_teammate = final_wr[final_wr['Teammate_player_display_name'].isna()]

null_players_teammate

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played


In [127]:
# View all rows where player_display_name is null
null_players_qb = final_wr[final_wr['QB_player_display_name'].isna()]

null_players_qb

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played


In [116]:
seasonal[seasonal['player_display_name'] == "Ja'Marr Chase"]

,player_display_name,season,completions_pg,attempts_pg,passing_yards_pg,passing_tds_pg,passing_interceptions_pg,sacks_suffered_pg,sack_yards_lost_pg,sack_fumbles_pg,sack_fumbles_lost_pg,passing_air_yards_pg,passing_yards_after_catch_pg,passing_first_downs_pg,passing_2pt_conversions_pg,passing_10_pg,passing_16_pg,passing_20_pg,passing_40_pg,passing_epa_pg,passing_cpoe_pg,carries_pg,rushing_yards_pg,rushing_tds_pg,rushing_fumbles_pg,rushing_fumbles_lost_pg,rushing_first_downs_pg,rushing_2pt_conversions_pg,rushing_10_pg,rushing_12_pg,rushing_20_pg,rushing_40_pg,rushing_epa_pg,receptions_pg,targets_pg,receiving_yards_pg,receiving_tds_pg,receiving_fumbles_pg,receiving_fumbles_lost_pg,receiving_air_yards_pg,receiving_yards_after_catch_pg,receiving_first_downs_pg,receiving_2pt_conversions_pg,receiving_10_pg,receiving_16_pg,receiving_20_pg,receiving_40_pg,air_yards_share_pg,wopr_pg,special_teams_tds_pg,punt_returns_pg,punt_return_yards_pg,kickoff_returns_pg,kickoff_return_yards_pg,fantasy_points_pg,fantasy_points_ppr_pg,games_played
4884,Ja'Marr Chase,2021,0.0000,0.0000,0.0000,0.0,0.0,0.000000,0.00,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.411765,1.235294,0.0,0.0,0.0,0.117647,0.0,0.058824,0.000000,0.0,0.0,-0.445415,4.764706,7.529412,85.588235,0.764706,0.117647,0.058824,95.117647,38.294118,3.294118,0.0,3.000000,1.882353,1.294118,0.470588,0.364920,0.603939,0.0,0.0,0.0,0.0,0.0,13.152941,17.917647,17
4885,Ja'Marr Chase,2022,0.0000,0.0000,0.0000,0.0,0.0,0.083333,-0.75,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.604014,NaN,0.416667,0.666667,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,-1.505549,7.250000,11.166667,87.166667,0.750000,0.166667,0.166667,101.666667,35.083333,4.833333,0.0,3.750000,1.416667,1.083333,0.250000,0.386652,0.713566,0.0,0.0,0.0,0.0,0.0,12.950000,20.200000,12
4886,Ja'Marr Chase,2023,0.0625,0.0625,-0.4375,0.0,0.0,0.000000,0.00,0.0,0.0,-0.4375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.102857,17.854065,0.187500,-0.375000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,-1.368989,6.250000,9.062500,76.000000,0.437500,0.062500,0.000000,78.375000,33.812500,3.937500,0.0,3.000000,1.375000,0.875000,0.250000,0.345560,0.632610,0.0,0.0,0.0,0.0,0.0,10.170000,16.420000,16
4887,Ja'Marr Chase,2024,0.0000,0.0000,0.0000,0.0,0.0,0.000000,0.00,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.176471,1.882353,0.0,0.0,0.0,0.117647,0.0,0.117647,0.058824,0.0,0.0,0.673911,7.470588,10.294118,100.470588,1.000000,0.000000,0.000000,89.764706,46.294118,4.411765,0.0,3.705882,1.882353,1.117647,0.470588,0.318476,0.631403,0.0,0.0,0.0,0.0,0.0,16.235294,23.705882,17
4888,Ja'Marr Chase,2025,0.0000,0.0000,0.0000,0.0,0.0,0.000000,0.00,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.187500,0.875000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.040955,7.812500,11.562500,88.250000,0.500000,0.062500,0.062500,98.125000,40.000000,4.562500,0.0,3.937500,1.625000,0.937500,0.125000,0.365829,0.737066,0.0,0.0,0.0,0.0,0.0,11.787500,19.600000,16


Names to fix: Josh Palmer, A.J. Green, A.J. Brown, K.J. Osborn, T.Y. Hilton, Robbie Chosen, C.J. Stroud, C.J. Beathard.

In [129]:
final_wr[final_wr["Previous_player_display_name"].isna()]

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
29,30,Ja'Marr Chase,2021,2020,21,CIN,Joe Burrow,2020,24,Tee Higgins,2020,22,Ja'Marr Chase,17.917647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Joe Burrow,2020,26.400000,40.400000,268.800000,1.300000,0.500000,3.200000,-23.100000,0.400000,0.300000,342.900000,114.200000,15.000000,0.000000,12.000000,4.600000,2.300000,0.300000,4.896672,1.898641,17.372000,10,Tee Higgins,2020,0.333333,1.866667,0.0,0.0,0.0,0.066667,0.0,0.066667,0.066667,0.0,0.0,0.252925,4.466667,7.200000,60.533333,0.400000,0.066667,0.066667,82.733333,20.666667,3.466667,0.0,2.600000,1.400000,0.933333,0.133333,0.276502,0.520051,0.0,0.000000,0.000000,0.0,0.0,12.973333,15
38,39,Jaylen Waddle,2021,2020,20,MIA,Tua Tagovailoa,2020,23,DeVante Parker,2020,24,Jaylen Waddle,15.487500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tua Tagovailoa,2020,18.600000,29.000000,181.400000,1.100000,0.500000,2.000000,-13.600000,0.100000,0.100000,217.500000,76.600000,10.000000,0.100000,7.600000,3.400000,1.500000,0.000000,-1.435119,2.113032,13.546000,10,DeVante Parker,2020,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,NaN,4.500000,7.357143,56.64

In [130]:
final_wr = final_wr[final_wr["Previous_player_display_name"].notna()].copy()

The code above removes some rows where rookies were mistakenly pulled into the dataset. This is problematic because then there is no previous season data for the model.

In [133]:
final_wr[final_wr["player_display_name"].isna()]

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
263,48,Brandon Aiyuk,2025,2024,27,SF,Brock Purdy,2024,25,Ricky Pearsall,2024,24,NaN,NaN,Brandon Aiyuk,2024.0,0.000000,0.000000,0.000000,0.000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0000,NaN,3.571429,6.714286,53.428571,0.000000,0.000000,0.000000,79.428571,15.285714,2.714286,0.000000,2.428571,1.142857,0.714286,0.142857,0.267098,0.519892,0.000000,0.000000,0.000000,0.000000,0.000000,8.914286,7.0,Brock Purdy,2024,20.000000,30.333333,257.600000,1.333333,0.800000,2.066667,-10.400000,0.333333,0.200000,260.266667,109.600000,11.866667,0.000000,10.400000,5.066667,3.733333,0.733333,5.185512,1.877466,17.790667,15,Ricky Pearsall,2024,0.272727,4.090909,0.000000,0.000000,0.000000,0.090909,0.0,0.090909,0.090909,0.090909,0.0000,1.103091,2.818182,4.181818,36.363636,0.272727,0.000000,0.000000,48.000000,10.363636,1.727273,0.000000,1.363636,0.545455,0.454545,0.181818,0.183686,0.320898,0.000000,0.272727,3.727273,0.000000,0.000000,8.500000,11
268,53,Diontae Johnson,2025,2024,29,BAL,Lamar Jackson,2024,28,Zay Flowers,2024,25,NaN,NaN,Diontae Johnson,2024.0,0.181818,0.545455,0.000000,0.000,0.0,0.090909,0.0,0.000000,0.000000,0.000000,0.0000,2.038883,3.000000,6.090909,34.090909,0.272727,0.000000,0.000000,66.454545,8.272727,1.909091,0.000000,1.545455,0.909091,0.454545,0.000000,0.324116,0.511485,0.000000,0.090909,0.000000,0.00000

In [134]:
mask = final_wr["player_display_name"].isna() & (final_wr["season"] < 2026)

final_wr.loc[mask, "player_display_name"] = final_wr.loc[mask, "player_name"]
final_wr.loc[mask, "fantasy_points_ppr_pg"] = 0

The code above imputes a zero for players that genuinely scored no fantasy points that season in Brandon Aiyuk and Diontae Johnson.

In [137]:
cols = [
    "Previous_rushing_epa_pg",
    "QB_passing_cpoe_pg",
    "Teammate_rushing_epa_pg"
]

final_wr[cols] = final_wr[cols].fillna(0)

Imputes a zero for any final null values.

In [138]:
final_wr.isnull().sum()

preseason_rank                              0
player_name                                 0
season                                      0
past_season                                 0
player_age                                  0
team                                        0
team_qb                                     0
team_qb_season                              0
team_qb_age                                 0
teammate                                    0
teammate_season                             0
teammate_age                                0
player_display_name                        56
fantasy_points_ppr_pg                      56
Previous_player_display_name                0
Previous_season                             0
Previous_carries_pg                         0
Previous_rushing_yards_pg                   0
Previous_rushing_tds_pg                     0
Previous_rushing_fumbles_pg                 0
Previous_rushing_fumbles_lost_pg            0
Previous_rushing_first_downs_pg   

The remaining nulls are just 2026 players that we don't know the result for yet in terms of PPR fantasy points per game.

#### Final Cleaned Dataset

In [139]:
final_wr.to_csv('../outputs/final_wrs_model_data_2020_2026.csv', index=False)

### Manual Input Dataset - RB

This dataset was put together from Claude and QA'd. It has the top 60 non-rookie RBs from 2021-2026, as well as that player's preseason projected QB and projected best WR teammate as well as their ages.

In [120]:
fantasy_board = pd.read_csv('../inputs/top60_preseason_rb_ppr_2021_2026.csv')

In [63]:
fantasy_board.head()

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age
0,1,Christian McCaffrey,2021,2020,25,CAR,Sam Darnold,2020,24,Royce Freeman,2020,25
1,2,Dalvin Cook,2021,2020,26,MIN,Kirk Cousins,2020,33,Alexander Mattison,2020,22
2,3,Alvin Kamara,2021,2020,26,NO,Jameis Winston,2020,27,Tony Jones,2020,24
3,4,Derrick Henry,2021,2020,27,TEN,Ryan Tannehill,2020,33,D'Onta Foreman,2020,25
4,5,Ezekiel Elliott,2021,2020,26,DAL,Dak Prescott,2020,28,Tony Pollard,2020,24


In [121]:
fantasy_board['player_name'] = (
    fantasy_board['player_name']
    .str.replace(r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$', '', regex=True)
)

fantasy_board['team_qb'] = (
    fantasy_board['team_qb']
    .str.replace(r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$', '', regex=True)
)

fantasy_board['teammate'] = (
    fantasy_board['teammate']
    .str.replace(r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$', '', regex=True)
)

In [122]:
temp_rb_1 = pd.merge(fantasy_board, target, left_on=['player_name', 'season'], 
                   right_on=['player_display_name', 'season'], how='left')

In [123]:
temp_rb_2 = pd.merge(temp_rb_1, player_previous_seasonal, left_on=['player_name', 'past_season'], 
                   right_on=['Previous_player_display_name', 'Previous_season'], how='left')

In [124]:
temp_rb_3 = pd.merge(temp_rb_2, qb_seasonal, left_on=['team_qb', 'team_qb_season'], 
                   right_on=['QB_player_display_name', 'QB_season'], how='left')

In [125]:
final_rb = pd.merge(temp_rb_3, teammate_seasonal, left_on=['teammate', 'teammate_season'], 
                   right_on=['Teammate_player_display_name', 'Teammate_season'], how='left')

In [50]:
final_rb.head()

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
0,1,Christian McCaffrey,2021,2020,25,CAR,Sam Darnold,2020,24,Royce Freeman,2020,25,Christian McCaffrey,18.214286,Christian McCaffrey,2020.0,19.666667,75.000000,1.666667,0.000000,0.000000,4.666667,0.000000,1.333333,1.000000,0.000000,0.000000,1.311211,5.666667,6.333333,49.666667,0.333333,0.000000,0.000000,9.000000,41.000000,2.666667,0.000000,1.666667,0.666667,0.333333,0.000000,0.032048,0.257394,0.0,0.000000,0.0,0.000000,0.000000,30.133333,3.0,Sam Darnold,2020.0,18.083333,30.333333,184.0000,0.7500,0.916667,2.916667,-19.500000,0.333333,0.166667,232.833333,96.250000,9.083333,0.083333,7.250000,3.833333,2.250,0.166667,-6.404763,-1.828693,11.168333,12.0,Royce Freeman,2020.0,3.181818,15.454545,0.000000,0.0,0.0,0.909091,0.0,0.454545,0.272727,0.181818,0.000,-0.026375,1.090909,1.181818,7.363636,0.000000,0.0,0.0,2.000000,5.727273,0.363636,0.0,0.272727,0.090909,0.090909,0.0,0.005844,0.053940,0.0,0.0,0.0,0.0,0.000,3.372727,11.0
1,2,Dalvin Cook,2021,2020,26,MIN,Kirk Cousins,2020,33,Alexander Mattison,2020,22,Dalvin Cook,15.869231,Dalvin Cook,2020.0,22.285714,111.214286,1.142857,0.285714,0.142857,6.500000,0.214286,3.285714,2.285714,0.428571,0.071429,1.153851,3.142857,3.857143,25.785714,0.071429,0.071429,0.071429,-6.857143,31.285714,1.214286,0.000000,1.000000,0.357143,0.285714,0.071429,-0.058278,0.158838,0.0,0.000000,0.0,0

**Data Cleaning**

Find players that did not have joined data because of formatting issues in names (capitalization, apostrophes, etc.) and change the name in the manual csv file. 

Once done, players with player_display_name NaN mean that they did not record any stats that season. Impute a zero for fantasy_points_ppr_pg.

Same with teammate_player_display_name.

Then rerun the isnull code and impute zeroes for anything remaining (not including fantasy points per game for 2026 rookies).

Gus Edwards (2021), ELijah Mitchell (2024), Joe Mixon (2025), and MarShawn Lloyd (2025) have missing values in the join because they did not play at all during the season. I will remove these rows.

In [126]:
# View all rows where player_display_name is null
null_players_teammate = final_rb[final_rb['Teammate_player_display_name'].isna()]

null_players_teammate

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
13,14,David Montgomery,2021,2020,24,CHI,Andy Dalton,2020,34,Damien Williams,2020,28,David Montgomery,15.0,David Montgomery,2020.0,16.466667,71.333333,0.533333,0.0000,0.0000,3.933333,0.066667,1.6000,0.8000,0.333333,0.133333,-0.600822,3.600,4.533333,29.20,0.133333,0.066667,0.066667,4.266667,26.4000,1.666667,0.0,1.333333,0.466667,0.133333,0.000,0.012908,0.188074,0.0,0.0,0.0,0.0,0.0,17.653333,15.0,Andy Dalton,2020.0,19.636364,30.272727,197.272727,1.272727,0.727273,2.181818,-16.727273,0.090909,0.000000,208.090909,94.818182,10.545455,0.000000,7.818182,2.818182,1.818182,0.454545,-1.018261,0.032265,12.445455,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,27,Raheem Mostert,2021,2020,29,SF,Jimmy Garoppolo,2020,29,Elijah Mitchell,2020,24,Raheem Mostert,2.0,Raheem Mostert,2020.0,13.000000,65.125000,0.250000,0.1250,0.1250,2.750000,0.000000,1.3750,0.8750,0.250000,0.125000,-1.328639,2.000,2.375000,19.50,0.125000,0.000000,0.000000,2.500000,17.0000,0.625000,0.0,0.375000,0.125000,0.125000,0.125,0.007858,0.109359,0.0,0.0,0.0,0.0,0.0,12.462500,8.0,Jimmy Garoppolo,2020.0,15.666667,23.333333,182.666667,1.166667,0.833333,1.833333,-12.833333,0.166667,0.000000,146.500000,119.000000,9.166667,0.000000,7.666667,3.666667,1.666

In [127]:
# View all rows where player_display_name is null
null_players_qb = final_rb[final_rb['QB_player_display_name'].isna()]

null_players_qb

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
43,44,Tevin Coleman,2021,2020,28,NYJ,Josh Johnson,2020,34,Ty Johnson,2020,24,Tevin Coleman,4.681818,Tevin Coleman,2020.0,4.666667,8.833333,0.000000,0.000000,0.000000,0.500000,0.0,0.333333,0.166667,0.000000,0.000000,-1.407482,0.666667,0.833333,5.666667,0.000000,0.000000,0.000000,2.166667,3.500000,0.333333,0.0,0.333333,0.166667,0.000000,0.0,0.013912,0.055787,0.0,0.0,0.0,0.000000,0.000000,2.116667,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ty Johnson,2020.0,4.909091,23.090909,0.090909,0.000000,0.000000,1.000000,0.0,0.363636,0.363636,0.090909,0.000000,0.119872,1.454545,1.909091,9.000000,0.090909,0.000000,0.000000,4.818182,11.727273,0.545455,0.0,0.363636,0.090909,0.000000,0.0,0.012004,0.112841,0.0,0.0,0.0,0.454545,8.545455,5.754545,11.0
176,17,Aaron Jones,2025,2024,30,MIN,JJ McCarthy,2024,22,Jordan Mason,2024,26,Aaron Jones,9.891667,Aaron Jones,2024.0,15.000000,66.941176,0.294118,0.176471,0.058824,2.882353,0.0,1.470588,1.176471,0.176471,0.058824,-0.790950,3.000000,3.647059,24.000000,0.117647,0.117647,0.117647,3.176471,24.176471,1.176471,0.0,1.000000,0.411765,0.235294,0.0,0.007210,0.187638,0.0,0.0,0.0,0.000000,0.000000,14.211765,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Jordan Mason,2024.0,12.750000,65.75

In [128]:
# View all rows where player_display_name is null
null_players_previous = final_rb[final_rb['Previous_player_display_name'].isna()]

null_players_previous

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
44,45,Damien Williams,2021,2020,28,CHI,Andy Dalton,2020,34,David Montgomery,2020,24,Damien Williams,5.058333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Andy Dalton,2020.0,19.636364,30.272727,197.272727,1.272727,0.727273,2.181818,-16.727273,0.090909,0.000000,208.090909,94.818182,10.545455,0.000000,7.818182,2.818182,1.818182,0.454545,-1.018261,0.032265,12.445455,11.0,David Montgomery,2020.0,16.466667,71.333333,0.533333,0.000000,0.000000,3.933333,0.066667,1.600000,0.800000,0.333333,0.133333,-0.600822,3.6,4.533333,29.200000,0.133333,0.066667,0.066667,4.266667,26.4,1.666667,0.0,1.333333,0.466667,0.133333,0.0,0.012908,0.188074,0.0,0.0,0.0,0.0,0.000000,17.653333,15.0
117,29,Odell Beckham,2023,2022,30,BAL,Lamar Jackson,2022,26,Gus Edwards,2022,28,Odell Beckham,7.678571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lamar Jackson,2022.0,13.533333,21.733333,149.466667,1.133333,0.466667,1.733333,-7.600000,0.066667,0.066667,183.600000,64.933333,7.000000,0.066667,6.266667,3.200000,1.600000,0.266667,0.873361,-1.451075,15.738667,15.0,Gus Edwards,2022.0,9.666667,48.111111,0.333333,0.111111,0.111111,3.333333,0.000000,1.

In [129]:
final_rb = final_rb[
    ~(final_rb["player_display_name"].isna() & (final_rb["season"] < 2026))
    & final_rb["Previous_player_display_name"].notna()
    & final_rb["Teammate_player_display_name"].notna()
    & final_rb["QB_player_display_name"].notna()
].copy()

The code above removed players that did not play in the previous season, or players whose teammates did not play in the previous season. Unlike the WRs, a team has less depth at RB, so I did not want to update a player's best teammate to someone who had no production, which could materially throw off projections.

In [130]:
final_rb[final_rb["player_display_name"].isna()]

,preseason_rank,player_name,season,past_season,player_age,team,team_qb,team_qb_season,team_qb_age,teammate,teammate_season,teammate_age,player_display_name,fantasy_points_ppr_pg,Previous_player_display_name,Previous_season,Previous_carries_pg,Previous_rushing_yards_pg,Previous_rushing_tds_pg,Previous_rushing_fumbles_pg,Previous_rushing_fumbles_lost_pg,Previous_rushing_first_downs_pg,Previous_rushing_2pt_conversions_pg,Previous_rushing_10_pg,Previous_rushing_12_pg,Previous_rushing_20_pg,Previous_rushing_40_pg,Previous_rushing_epa_pg,Previous_receptions_pg,Previous_targets_pg,Previous_receiving_yards_pg,Previous_receiving_tds_pg,Previous_receiving_fumbles_pg,Previous_receiving_fumbles_lost_pg,Previous_receiving_air_yards_pg,Previous_receiving_yards_after_catch_pg,Previous_receiving_first_downs_pg,Previous_receiving_2pt_conversions_pg,Previous_receiving_10_pg,Previous_receiving_16_pg,Previous_receiving_20_pg,Previous_receiving_40_pg,Previous_air_yards_share_pg,Previous_wopr_pg,Previous_special_teams_tds_pg,Previous_punt_returns_pg,Previous_punt_return_yards_pg,Previous_kickoff_returns_pg,Previous_kickoff_return_yards_pg,Previous_fantasy_points_ppr_pg,Previous_games_played,QB_player_display_name,QB_season,QB_completions_pg,QB_attempts_pg,QB_passing_yards_pg,QB_passing_tds_pg,QB_passing_interceptions_pg,QB_sacks_suffered_pg,QB_sack_yards_lost_pg,QB_sack_fumbles_pg,QB_sack_fumbles_lost_pg,QB_passing_air_yards_pg,QB_passing_yards_after_catch_pg,QB_passing_first_downs_pg,QB_passing_2pt_conversions_pg,QB_passing_10_pg,QB_passing_16_pg,QB_passing_20_pg,QB_passing_40_pg,QB_passing_epa_pg,QB_passing_cpoe_pg,QB_fantasy_points_ppr_pg,QB_games_played,Teammate_player_display_name,Teammate_season,Teammate_carries_pg,Teammate_rushing_yards_pg,Teammate_rushing_tds_pg,Teammate_rushing_fumbles_pg,Teammate_rushing_fumbles_lost_pg,Teammate_rushing_first_downs_pg,Teammate_rushing_2pt_conversions_pg,Teammate_rushing_10_pg,Teammate_rushing_12_pg,Teammate_rushing_20_pg,Teammate_rushing_40_pg,Teammate_rushing_epa_pg,Teammate_receptions_pg,Teammate_targets_pg,Teammate_receiving_yards_pg,Teammate_receiving_tds_pg,Teammate_receiving_fumbles_pg,Teammate_receiving_fumbles_lost_pg,Teammate_receiving_air_yards_pg,Teammate_receiving_yards_after_catch_pg,Teammate_receiving_first_downs_pg,Teammate_receiving_2pt_conversions_pg,Teammate_receiving_10_pg,Teammate_receiving_16_pg,Teammate_receiving_20_pg,Teammate_receiving_40_pg,Teammate_air_yards_share_pg,Teammate_wopr_pg,Teammate_special_teams_tds_pg,Teammate_punt_returns_pg,Teammate_punt_return_yards_pg,Teammate_kickoff_returns_pg,Teammate_kickoff_return_yards_pg,Teammate_fantasy_points_ppr_pg,Teammate_games_played
212,1,Jahmyr Gibbs,2026,2025,24,DET,Jared Goff,2025,31,Isiah Pacheco,2025,27,NaN,NaN,Jahmyr Gibbs,2025.0,14.294118,71.941176,0.764706,0.117647,0.058824,3.294118,0.000000,1.588235,1.235294,0.588235,0.352941,-0.156885,4.529412,5.529412,36.235294,0.294118,0.000000,0.000000,3.176471,36.176471,1.647059,0.000000,1.352941,0.588235,0.411765,0.058824,0.014739,0.252620,0.000000,0.000000,0.000000,0.000000,0.000000,21.582353,17.0,Jared Goff,2025.0,23.117647,34.000000,268.470588,2.000000,0.470588,2.235294,-15.235294,0.176471,0.176471,218.529412,142.470588,13.117647,0.000000,10.882353,5.941176,3.882353,0.470588,6.262362,2.889624,17.474118,17.0,Isiah Pacheco,2025.0,9.076923,35.538462,0.076923,0.000000,0.000000,2.076923,0.000000,0.769231,0.307692,0.000000,0.000000,-1.049492,1.461538,2.000000,7.769231,0.076923,0.000000,0.000000,-2.076923,9.307692,0.307692,0.000000,0.230769,0.076923,0.076923,0.000000,-0.006067,0.097735,0.000000,0.000000,0.000000,0.000000,0.000000,6.715385,13.0
213,2,Bijan Robinson,2026,2025,24,ATL,Michael Penix,2025,26,Brian Robinson,2025,27,NaN,NaN,Bijan Robinson,2025.0,16.882353,86.941176,0.411765,0.176471,0.117647,3.705882,0.058824,2.117647,1.411765,0.470588,0.117647,-0.720861,4.647059,6.058824,48.235294,0.235294,0.058824,0.058824,5.941176,50.411765,2.000000,0.000000,1.588235,0.764706,0.588235,0.235294,

In [132]:
cols = [
    "QB_passing_cpoe_pg",
    "Teammate_rushing_epa_pg"
]

final_rb[cols] = final_rb[cols].fillna(0)

Imputes a zero for any final null values.

In [133]:
final_rb.isnull().sum()

preseason_rank                              0
player_name                                 0
season                                      0
past_season                                 0
player_age                                  0
team                                        0
team_qb                                     0
team_qb_season                              0
team_qb_age                                 0
teammate                                    0
teammate_season                             0
teammate_age                                0
player_display_name                        51
fantasy_points_ppr_pg                      51
Previous_player_display_name                0
Previous_season                             0
Previous_carries_pg                         0
Previous_rushing_yards_pg                   0
Previous_rushing_tds_pg                     0
Previous_rushing_fumbles_pg                 0
Previous_rushing_fumbles_lost_pg            0
Previous_rushing_first_downs_pg   

The remaining nulls are just 2026 players that we don't know the result for yet in terms of PPR fantasy points per game.

### Final Cleaned Dataset

In [134]:
final_rb.to_csv('../outputs/final_rbs_model_data_2020_2026.csv', index=False)